In [ ]:
import matplotlib.pyplot as plt

from utils.data import DataManager
from utils.tools.config import (
    BENCHMARKS, 
    RISK_ANALYSIS,
    ANALYSIS_START_DATE,
    ANALYSIS_END_DATE
)

from utils.analysis.capm import (
    CAPMAnalyzer,
    PortfolioOptimizationAnalyzer,
    MultiAssetCAPMAnalyzer,
    CAPMReporter,
    PortfolioReporter,
    MultiAssetReporter
)

from utils.visualizations import (
    CAPMVisualizer,
    PortfolioOptimizationVisualizer,
    MultiAssetCAPMVisualizer
)

In [ ]:
# ANALYSIS CONFIGURATION
# Dates come from config.py (ANALYSIS_DATES)
# Customize here only if you need different values

# Portfolio to analyze
TICKERS = ["SYF", "CF", "AFL", "NEM"]
BENCHMARK_NAME = "SP500"

# (Optional) Custom dates - Leave empty to use config.py
USE_CUSTOM_DATES = False
START_DATE = ""  # e.g.: "2020-01-01"
END_DATE = ""    # e.g.: "2024-12-31"

# Constants from config
BENCHMARK_TICKER = BENCHMARKS[BENCHMARK_NAME]
RISK_FREE_RATE = RISK_ANALYSIS['risk_free_rate']
ANNUAL_FACTOR = RISK_ANALYSIS['annual_factor']

# Resolve dates
if USE_CUSTOM_DATES and START_DATE and END_DATE:
    final_start, final_end = START_DATE, END_DATE
    print(f"Using custom dates: {final_start} -> {final_end}")
else:
    final_start, final_end = ANALYSIS_START_DATE, ANALYSIS_END_DATE
    print(f"Using config.py dates: {final_start} -> {final_end}")

print(f"\nPortfolio: {len(TICKERS)} assets")
print(f"Benchmark: {BENCHMARK_NAME} ({BENCHMARK_TICKER})")
print(f"Risk-Free Rate: {RISK_FREE_RATE:.2%}")

In [ ]:
# INITIALIZATION
# Create shared DataManager
data_manager = DataManager()

# Initialize analyzers
capm_analyzer = CAPMAnalyzer(annual_factor=ANNUAL_FACTOR)
portfolio_optimizer = PortfolioOptimizationAnalyzer(annual_factor=ANNUAL_FACTOR)
multi_capm = MultiAssetCAPMAnalyzer(annual_factor=ANNUAL_FACTOR)

# Initialize reporters
capm_reporter = CAPMReporter(capm_analyzer)
portfolio_reporter = PortfolioReporter(portfolio_optimizer)
multi_asset_reporter = MultiAssetReporter(multi_capm)

# Initialize visualizers
capm_viz = CAPMVisualizer(capm_analyzer)
portfolio_viz = PortfolioOptimizationVisualizer(portfolio_optimizer)
multi_asset_viz = MultiAssetCAPMVisualizer(multi_capm)

print("[OK] Analyzers, reporters and visualizers initialized")

In [ ]:
# DATA DOWNLOAD
print("\nDownloading data...")

assets_prices, benchmark_prices = data_manager.download_portfolio_with_benchmark(
    tickers=TICKERS,
    benchmark_name=BENCHMARK_NAME,
    start_date=final_start,
    end_date=final_end
)

# Calculate returns
returns = assets_prices.pct_change().dropna()
benchmark_returns = benchmark_prices.pct_change().dropna()

print(f"\n[OK] Data downloaded:")
print(f"   Period: {assets_prices.index[0].date()} -> {assets_prices.index[-1].date()}")
print(f"   Days: {len(assets_prices)}")
print(f"   Assets: {list(returns.columns)}")

In [ ]:
# INDIVIDUAL CAPM ANALYSIS
# Analyze a specific asset vs the market

ASSET_TO_ANALYZE = "META"  # Change ticker here

# If the asset is not in returns, download it
if ASSET_TO_ANALYZE not in returns.columns:
    print(f"Downloading data for {ASSET_TO_ANALYZE}...")
    asset_prices = data_manager.download_assets(
        tickers=[ASSET_TO_ANALYZE],
        start_date=final_start,
        end_date=final_end
    )
    asset_returns = asset_prices[ASSET_TO_ANALYZE].pct_change().dropna()
    print(f"[OK] Data downloaded for {ASSET_TO_ANALYZE}")
else:
    asset_returns = returns[ASSET_TO_ANALYZE]

print(f"\nAnalyzing {ASSET_TO_ANALYZE} with CAPM...\n")

capm_reporter.generate_report(
    asset_returns=asset_returns.values,
    market_returns=benchmark_returns.values,
    risk_free_rate=RISK_FREE_RATE,
    asset_name=ASSET_TO_ANALYZE
)

In [ ]:
# 📊 VISUALIZACIÓN CAPM INDIVIDUAL
# Usar asset_returns si fue descargado, sino usar returns[ASSET_TO_ANALYZE]
if ASSET_TO_ANALYZE not in returns.columns:
    asset_returns_for_viz = asset_returns
else:
    asset_returns_for_viz = returns[ASSET_TO_ANALYZE]

fig1 = capm_viz.plot_capm_analysis(
    asset_returns=asset_returns_for_viz.values,
    market_returns=benchmark_returns.values,
    risk_free_rate=RISK_FREE_RATE,
    asset_name=ASSET_TO_ANALYZE
)
plt.show()

In [ ]:
# EFFICIENT FRONTIER AND CML
print("\nCalculating efficient frontier...\n")

fig2 = portfolio_viz.plot_efficient_frontier_analysis(
    returns=returns,
    risk_free_rate=RISK_FREE_RATE
)
plt.show()

In [ ]:
# TANGENT PORTFOLIO (Maximum Sharpe Ratio)
print("\nCalculating tangent portfolio...\n")

portfolio_reporter.generate_tangent_report(
    returns=returns,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# MULTI-ASSET CAPM ANALYSIS (Summary)
print("\nAnalyzing all assets with CAPM...\n")

multi_asset_reporter.generate_summary_report(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

In [ ]:
# CAPM RESULTS TABLE
capm_results = multi_capm.analyze_multiple(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)

print("\nDetailed CAPM results:\n")
display(capm_results)

In [ ]:
# MULTI-ASSET VISUALIZATION
fig3 = multi_asset_viz.plot_multi_asset_analysis(
    returns=returns,
    market_returns=benchmark_returns,
    risk_free_rate=RISK_FREE_RATE
)
plt.show()

In [ ]:
# INDIVIDUAL CAPM ANALYSIS
print("CAPM analysis per asset:\n")
print(f"{'Ticker':<8} {'Alpha':<10} {'Beta':<8} {'R2':<8} {'Sig':<5}")
print("-" * 50)

for ticker in TICKERS:
    capm_result = capm_analyzer.analyze(
        asset_returns=returns[ticker].values,
        market_returns=benchmark_returns.values,
        risk_free_rate=RISK_FREE_RATE
    )
    
    is_sig = "[OK]" if capm_result['is_significant'] else "[--]"
    
    print(f"{ticker:<8} {capm_result['alpha_annual']:>8.2%}  "
          f"{capm_result['beta']:>6.3f}  "
          f"{capm_result['r_squared']:>6.3f}  {is_sig:<5}")